# 7.4 端侧推理服务

## 目的

单次 `generate()` 不够：手机上可能同时有助手、输入法、翻译在抢模型。需要 **排队、优先级、多模型内存、超时降级、热更新**。

配套监控见 [7.4_edge_monitoring.ipynb](7.4_edge_monitoring.ipynb)。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass, field
from typing import Optional
import time
import hashlib
import hmac
import json
import math

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch {torch.__version__}")

## 7.4.1 请求优先级队列

In [ ]:
from enum import IntEnum
import heapq


class Priority(IntEnum):
    BACKGROUND = 0
    NORMAL = 1
    INTERACTIVE = 2
    CRITICAL = 3


@dataclass(order=True)
class InferRequest:
    sort_key: tuple = field(init=False, repr=False)
    priority: Priority = field(compare=False)
    req_id: str = field(compare=False)
    prompt_tokens: int = field(compare=False)
    deadline_ms: float = field(compare=False)
    enqueued_at: float = field(compare=False, default_factory=time.perf_counter)

    def __post_init__(self):
        # 高优先级先出；同优先级 FIFO（用入队时间）
        self.sort_key = (-int(self.priority), self.enqueued_at)


class RequestQueue:
    def __init__(self):
        self._hq = []

    def push(self, req: InferRequest):
        heapq.heappush(self._hq, req)

    def pop(self) -> Optional[InferRequest]:
        return heapq.heappop(self._hq) if self._hq else None

    def __len__(self):
        return len(self._hq)


q = RequestQueue()
for r in [
    InferRequest(Priority.BACKGROUND, "bg-summarize", 512, 5000),
    InferRequest(Priority.INTERACTIVE, "chat", 64, 800),
    InferRequest(Priority.CRITICAL, "voice-wakeup", 32, 300),
    InferRequest(Priority.NORMAL, "translate", 128, 1500),
]:
    q.push(r)

print("出队顺序:")
while len(q):
    r = q.pop()
    print(f"  {r.priority.name:12s} {r.req_id}")

## 7.4.2 多模型内存管理（驻留 / 换出）

In [ ]:
@dataclass
class ModelSlot:
    name: str
    size_mb: float
    last_used: float = 0.0
    pinned: bool = False


class ModelMemoryManager:
    def __init__(self, budget_mb: float):
        self.budget = budget_mb
        self.loaded: dict[str, ModelSlot] = {}

    @property
    def used(self) -> float:
        return sum(m.size_mb for m in self.loaded.values())

    def ensure(self, name: str, size_mb: float, pinned: bool = False) -> list[str]:
        evicted = []
        now = time.perf_counter()
        if name in self.loaded:
            self.loaded[name].last_used = now
            return evicted
        while self.used + size_mb > self.budget:
            # LRU 驱逐未 pin 的模型
            candidates = [m for m in self.loaded.values() if not m.pinned]
            if not candidates:
                raise MemoryError(f"无法为 {name} 腾出 {size_mb}MB")
            victim = min(candidates, key=lambda m: m.last_used)
            del self.loaded[victim.name]
            evicted.append(victim.name)
        self.loaded[name] = ModelSlot(name, size_mb, now, pinned)
        return evicted


mm = ModelMemoryManager(budget_mb=2400)
print("load base:", mm.ensure("llm-1.5b", 1100, pinned=True))
print("load asr:", mm.ensure("whisper-tiny", 150))
print("load tts:", mm.ensure("tts-vits", 120))
print("resident before image gen:", sorted(mm.loaded), "used=", round(mm.used, 1))
try:
    print("load sd-turbo:", mm.ensure("sd-turbo", 1400))
except MemoryError as e:
    print("expected:", e)
    print("unpin base LLM to free headroom for image model")
    mm.loaded["llm-1.5b"].pinned = False
    print("load sd-turbo:", mm.ensure("sd-turbo", 1400))
print("resident:", sorted(mm.loaded), "used=", round(mm.used, 1))


## 7.4.3 超时降级与端云回退

In [ ]:
@dataclass
class ServePolicy:
    primary: str = "3B-W4"
    fallback: str = "1.5B-W4"
    cloud: str = "cloud-70B"
    soft_timeout_ms: float = 600
    hard_timeout_ms: float = 1200


def simulate_latency(model: str, tokens: int) -> float:
    speed = {"3B-W4": 22, "1.5B-W4": 35, "cloud-70B": 60}
    # prefill 固定开销 + decode
    return 80 + tokens / speed[model] * 1000


def serve(req_tokens: int, policy: ServePolicy, network_ok: bool = True) -> dict:
    for model in (policy.primary, policy.fallback):
        lat = simulate_latency(model, req_tokens)
        if lat <= policy.soft_timeout_ms or (model == policy.fallback and lat <= policy.hard_timeout_ms):
            return {"model": model, "latency_ms": round(lat, 1), "degraded": model != policy.primary}
    if network_ok:
        lat = simulate_latency(policy.cloud, req_tokens)
        return {"model": policy.cloud, "latency_ms": round(lat, 1), "degraded": True, "cloud": True}
    return {"model": None, "error": "timeout_offline"}


pol = ServePolicy()
for n in (16, 48, 128):
    print(f"tokens={n:3d} -> {serve(n, pol)}")
print("offline long:", serve(128, pol, network_ok=False))

## 7.4.4 热更新双缓冲 + 简易 A/B

In [ ]:
class HotSwapRuntime:
    def __init__(self, active: str):
        self.active = active
        self.staging: Optional[str] = None
        self.ab = {"A": active, "B": None}
        self.traffic_b = 0.0  # B 流量比例

    def stage(self, new_model: str):
        self.staging = new_model
        print(f"staging {new_model} in background...")

    def commit(self):
        assert self.staging
        self.active = self.staging
        self.staging = None
        print(f"atomic swap -> {self.active}")

    def enable_ab(self, b_model: str, traffic_b: float):
        self.ab["B"] = b_model
        self.traffic_b = traffic_b

    def pick(self, user_hash: int) -> str:
        if self.ab["B"] and (user_hash % 100) < self.traffic_b * 100:
            return self.ab["B"]
        return self.ab["A"] or self.active


rt = HotSwapRuntime("qwen-1.5b-v1")
rt.stage("qwen-1.5b-v2")
rt.commit()
rt.enable_ab("qwen-1.5b-v2-lora", traffic_b=0.2)
picks = [rt.pick(h) for h in range(100)]
print("A/B sample:", {k: picks.count(k) for k in set(picks)})

## 小结

1. 交互请求永远优先于后台总结。
2. 内存预算内 LRU + pin 基座模型。
3. soft/hard timeout 触发小模型或云端。
4. 热更新用双缓冲；灰度用 A/B 流量切片。